# **Norwegian News Sources**

Project for TDT4310

By: Malin Haugland Høli

## XML Parsing

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import re

# --- Utility functions ---
def parse_article(path):
    """
    Parse a single XML article and return a dict of all relevant info.
    """
    tree = ET.parse(path)
    root = tree.getroot()

    # --- HEADER ---
    header = root.find("header")
    attrs = {attr.attrib["name"]: attr.attrib.get("value", "") for attr in header.findall("attribute")}

    # Derived field: is author empty?
    author_empty = attrs.get("author", "") == ""

    # --- BODY ---
    body = root.find("body")
    title = None
    ingress = None
    paragraphs = []

    for div in body.findall("div"):
        div_type = div.attrib.get("type")
        if div_type == "title" and div.attrib.get("level") == "1":
            title = div.text
        elif div_type == "ingress":
            ingress = div.text
        elif div_type == "text":
            for p in div.findall("p"):
                if p.text:
                    paragraphs.append(p.text)

    full_text = "\n".join(paragraphs)

    # Derived features
    word_count = len(re.findall(r"\w+", full_text))
    sentence_count = len(re.findall(r"[.!?]+", full_text))

    return {
        **attrs,
        "author_empty": author_empty,
        "title": title,
        "ingress": ingress,
        "text": full_text,
        "word_count": word_count,
        "sentence_count": sentence_count
    }


def process_folder(folder_path, output_file):
    """
    Process all XML articles in a folder and save as Parquet.
    """
    rows = []
    for file in os.listdir(folder_path):
        path = os.path.join(folder_path, file)
        if not file.lower().endswith(".xml"):
            continue
        try:
            data = parse_article(path)
            data["file"] = file
            rows.append(data)
        except Exception as e:
            print(f"Error parsing {file}: {e}")

    df = pd.DataFrame(rows)
    df.to_parquet(output_file, index=False)
    print(f"Saved {len(df)} articles to {output_file}")


# --- Main pipeline ---
base_data_path = "data"       # folder containing XML folders
processed_path = "processed"  # output folder for Parquet files
os.makedirs(processed_path, exist_ok=True)

for folder in os.listdir(base_data_path):
    folder_path = os.path.join(base_data_path, folder)
    if os.path.isdir(folder_path):
        output_file = os.path.join(processed_path, f"{folder}.parquet")
        process_folder(folder_path, output_file)

Saved 25 articles to processed\aa-2019-nno.parquet
Saved 11618 articles to processed\aa-2019-nob.parquet
Saved 97 articles to processed\ap-2019-nno.parquet
Saved 27341 articles to processed\ap-2019-nob.parquet
Saved 820 articles to processed\bt-2019-nno.parquet
Saved 13017 articles to processed\bt-2019-nob.parquet
Saved 45 articles to processed\da-2019-nno.parquet
Saved 13065 articles to processed\da-2019-nob.parquet
Saved 65 articles to processed\db-2019-nno.parquet
Saved 23756 articles to processed\db-2019-nob.parquet
Saved 24 articles to processed\dn-2019-nno.parquet
Saved 13218 articles to processed\dn-2019-nob.parquet
Saved 43 articles to processed\fv-2019-nno.parquet
Saved 11418 articles to processed\fv-2019-nob.parquet
Saved 70 articles to processed\nl-2019-nno.parquet
Saved 10773 articles to processed\nl-2019-nob.parquet
Saved 821 articles to processed\sa-2019-nno.parquet
Saved 30729 articles to processed\sa-2019-nob.parquet
Saved 35 articles to processed\vg-2019-nno.parquet
Sa